In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EMP07 - OUTSIDE BUSINESS ACTIVITIES (OBA)
   ===================================================================================
   BUSINESS QUESTION:
   How many distinct employees were associated with high-risk outside business
   activities or consulting services during the assessment period, by Assessable
   Unit?

   DATA SOURCES:
      - hive_metastore.ra_adido_2025.replace_with_emp07_wealth_oba_table
      - hive_metastore.ra_adido_2025.replace_with_emp07_banking_oba_table
      - vw_cost_center_mapping_bootstrap
      - hive_metastore.ra_adido_2025.fy25_list_of_units

   IMPLEMENTATION NOTES:
      1. Replace the source tables and placeholder column names before running.
      2. Confirm the distinct employee key used in both files and keep
         COUNT(DISTINCT Employee_Key) at the AU level.
      3. Matching is intended to be case-insensitive keyword containment.
      4. Wealth logic uses Nature OR Duties. Banking logic uses Nature, except
         when Nature = 'Other', in which case Duties is the fallback text.

   QUERY LOGIC & SUMMARY:
      1. MASTER AU ANCHOR: Restrict final output to the standard ABAC AU list
         (Canada, Hong Kong, Barbados plus US_OR_CANADA = 'CANADA').
      2. KEYWORD SET: Build the approved high-risk OBA keyword list once and reuse
         it in both source branches.
      3. SOURCE BRANCHING:
         - Wealth: flag when either the business/entity field or the duties field
           contains any high-risk keyword.
         - Banking: flag when the business/entity field contains any keyword, or
           when that field is exactly 'Other' and the duties field contains any
           keyword.
      4. COST CENTER NORMALIZATION: Normalize the source-side Cost Center using
         the shared canonical rule before joining to the mapping bootstrap view.
      5. AU BRIDGE: Union the distinct flagged employees from both branches and map
         them to AU_ID via vw_cost_center_mapping_bootstrap.
      6. AGGREGATION: Count DISTINCT Employee_Key per AU.
      7. FALLBACK: LEFT JOIN the counts back to Master_AUs so all in-scope AUs
         return a row and default missing counts to 0.
=================================================================================== */

WITH Master_AUs AS (
    SELECT
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE
            WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING)))
        END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
      AND Cost_Center_ID IS NOT NULL
),

High_Risk_Keywords AS (
    SELECT * FROM VALUES
        ('financial services'),
        ('banking services'),
        ('insurance services'),
        ('insurance sales'),
        ('mortgage services'),
        ('investment services'),
        ('financial planning services'),
        ('investment advice'),
        ('fintech'),
        ('real estate brokerage'),
        ('real estate sales'),
        ('investment committee'),
        ('government'),
        ('government services'),
        ('religion'),
        ('religious services'),
        ('legal services'),
        ('appraisal services'),
        ('td vendor'),
        ('td supplier')
    AS High_Risk_Keywords(keyword)
),

Wealth_Source AS (
    SELECT
        TRIM(CAST(REPLACE_WITH_WEALTH_EMPLOYEE_KEY AS STRING)) AS Employee_Key,
        CASE
            WHEN TRIM(CAST(REPLACE_WITH_WEALTH_COST_CENTER AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(REPLACE_WITH_WEALTH_COST_CENTER AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(REPLACE_WITH_WEALTH_COST_CENTER AS STRING)))
        END AS Cleaned_CC,
        LOWER(TRIM(CAST(REPLACE_WITH_WEALTH_NATURE_TEXT AS STRING))) AS Nature_Text,
        LOWER(TRIM(CAST(REPLACE_WITH_WEALTH_DUTIES_TEXT AS STRING))) AS Duties_Text
    FROM hive_metastore.ra_adido_2025.replace_with_emp07_wealth_oba_table
    WHERE REPLACE_WITH_WEALTH_EMPLOYEE_KEY IS NOT NULL
      AND REPLACE_WITH_WEALTH_COST_CENTER IS NOT NULL
),

Wealth_Flagged AS (
    SELECT DISTINCT
        w.Employee_Key,
        w.Cleaned_CC
    FROM Wealth_Source w
    WHERE EXISTS (
        SELECT 1
        FROM High_Risk_Keywords k
        WHERE w.Nature_Text LIKE CONCAT('%', k.keyword, '%')
           OR w.Duties_Text LIKE CONCAT('%', k.keyword, '%')
    )
),

Banking_Source AS (
    SELECT
        TRIM(CAST(REPLACE_WITH_BANKING_EMPLOYEE_KEY AS STRING)) AS Employee_Key,
        CASE
            WHEN TRIM(CAST(REPLACE_WITH_BANKING_COST_CENTER AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(REPLACE_WITH_BANKING_COST_CENTER AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(REPLACE_WITH_BANKING_COST_CENTER AS STRING)))
        END AS Cleaned_CC,
        LOWER(TRIM(CAST(REPLACE_WITH_BANKING_NATURE_TEXT AS STRING))) AS Nature_Text,
        LOWER(TRIM(CAST(REPLACE_WITH_BANKING_DUTIES_TEXT AS STRING))) AS Duties_Text
    FROM hive_metastore.ra_adido_2025.replace_with_emp07_banking_oba_table
    WHERE REPLACE_WITH_BANKING_EMPLOYEE_KEY IS NOT NULL
      AND REPLACE_WITH_BANKING_COST_CENTER IS NOT NULL
),

Banking_Flagged AS (
    SELECT DISTINCT
        b.Employee_Key,
        b.Cleaned_CC
    FROM Banking_Source b
    WHERE EXISTS (
        SELECT 1
        FROM High_Risk_Keywords k
        WHERE b.Nature_Text LIKE CONCAT('%', k.keyword, '%')
           OR (
                b.Nature_Text = 'other'
            AND b.Duties_Text LIKE CONCAT('%', k.keyword, '%')
           )
    )
),

Combined_Flagged_OBAs AS (
    SELECT Employee_Key, Cleaned_CC FROM Wealth_Flagged
    UNION
    SELECT Employee_Key, Cleaned_CC FROM Banking_Flagged
),

Bridged_OBAs AS (
    SELECT DISTINCT
        m.AU_ID,
        c.Employee_Key,
        c.Cleaned_CC
    FROM Combined_Flagged_OBAs c
    INNER JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
),

AU_Employee_Counts AS (
    SELECT
        AU_ID,
        COUNT(DISTINCT Employee_Key) AS Flagged_Employee_Count
    FROM Bridged_OBAs
    GROUP BY AU_ID
)

SELECT
    a.BusinessID,
    a.AU_Name,
    a.Business_Segment,
    'EMP07' AS QuestionID,
    COALESCE(CAST(e.Flagged_Employee_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN AU_Employee_Counts e
    ON a.BusinessID = e.AU_ID
ORDER BY a.BusinessID;


In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EMP07 - AU Level Employee Routing Review
   PURPOSE: One row per AU showing normalized Cost Centers, distinct flagged
   employee counts, and whether the count came from Wealth, Banking, or both.
=================================================================================== */

WITH Master_AUs AS (
    SELECT
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE
            WHEN TRIM(CAST(Cost_Center_ID AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(Cost_Center_ID AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(Cost_Center_ID AS STRING)))
        END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
      AND Cost_Center_ID IS NOT NULL
),

High_Risk_Keywords AS (
    SELECT * FROM VALUES
        ('financial services'),
        ('banking services'),
        ('insurance services'),
        ('insurance sales'),
        ('mortgage services'),
        ('investment services'),
        ('financial planning services'),
        ('investment advice'),
        ('fintech'),
        ('real estate brokerage'),
        ('real estate sales'),
        ('investment committee'),
        ('government'),
        ('government services'),
        ('religion'),
        ('religious services'),
        ('legal services'),
        ('appraisal services'),
        ('td vendor'),
        ('td supplier')
    AS High_Risk_Keywords(keyword)
),

Wealth_Source AS (
    SELECT
        TRIM(CAST(REPLACE_WITH_WEALTH_EMPLOYEE_KEY AS STRING)) AS Employee_Key,
        CASE
            WHEN TRIM(CAST(REPLACE_WITH_WEALTH_COST_CENTER AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(REPLACE_WITH_WEALTH_COST_CENTER AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(REPLACE_WITH_WEALTH_COST_CENTER AS STRING)))
        END AS Cleaned_CC,
        LOWER(TRIM(CAST(REPLACE_WITH_WEALTH_NATURE_TEXT AS STRING))) AS Nature_Text,
        LOWER(TRIM(CAST(REPLACE_WITH_WEALTH_DUTIES_TEXT AS STRING))) AS Duties_Text
    FROM hive_metastore.ra_adido_2025.replace_with_emp07_wealth_oba_table
    WHERE REPLACE_WITH_WEALTH_EMPLOYEE_KEY IS NOT NULL
      AND REPLACE_WITH_WEALTH_COST_CENTER IS NOT NULL
),

Wealth_Flagged AS (
    SELECT DISTINCT
        'Wealth' AS Source_Name,
        w.Employee_Key,
        w.Cleaned_CC
    FROM Wealth_Source w
    WHERE EXISTS (
        SELECT 1
        FROM High_Risk_Keywords k
        WHERE w.Nature_Text LIKE CONCAT('%', k.keyword, '%')
           OR w.Duties_Text LIKE CONCAT('%', k.keyword, '%')
    )
),

Banking_Source AS (
    SELECT
        TRIM(CAST(REPLACE_WITH_BANKING_EMPLOYEE_KEY AS STRING)) AS Employee_Key,
        CASE
            WHEN TRIM(CAST(REPLACE_WITH_BANKING_COST_CENTER AS STRING)) RLIKE '^[0-9]+$'
            THEN LPAD(RIGHT(TRIM(CAST(REPLACE_WITH_BANKING_COST_CENTER AS STRING)), 4), 4, '0')
            ELSE UPPER(TRIM(CAST(REPLACE_WITH_BANKING_COST_CENTER AS STRING)))
        END AS Cleaned_CC,
        LOWER(TRIM(CAST(REPLACE_WITH_BANKING_NATURE_TEXT AS STRING))) AS Nature_Text,
        LOWER(TRIM(CAST(REPLACE_WITH_BANKING_DUTIES_TEXT AS STRING))) AS Duties_Text
    FROM hive_metastore.ra_adido_2025.replace_with_emp07_banking_oba_table
    WHERE REPLACE_WITH_BANKING_EMPLOYEE_KEY IS NOT NULL
      AND REPLACE_WITH_BANKING_COST_CENTER IS NOT NULL
),

Banking_Flagged AS (
    SELECT DISTINCT
        'Banking' AS Source_Name,
        b.Employee_Key,
        b.Cleaned_CC
    FROM Banking_Source b
    WHERE EXISTS (
        SELECT 1
        FROM High_Risk_Keywords k
        WHERE b.Nature_Text LIKE CONCAT('%', k.keyword, '%')
           OR (
                b.Nature_Text = 'other'
            AND b.Duties_Text LIKE CONCAT('%', k.keyword, '%')
           )
    )
),

Combined_Flagged_OBAs AS (
    SELECT * FROM Wealth_Flagged
    UNION
    SELECT * FROM Banking_Flagged
),

Bridged_OBAs AS (
    SELECT DISTINCT
        m.AU_ID,
        c.Source_Name,
        c.Employee_Key,
        c.Cleaned_CC
    FROM Combined_Flagged_OBAs c
    INNER JOIN CC_Mapping m
        ON c.Cleaned_CC = m.Mapped_CC
),

AU_Employee_Counts AS (
    SELECT
        AU_ID,
        COUNT(DISTINCT Employee_Key) AS Included_Employee_Count,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Source_Name))) AS Flagged_Source_List
    FROM Bridged_OBAs
    GROUP BY AU_ID
),

AU_Cost_Center_List AS (
    SELECT
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_Cost_Centers
    FROM Bridged_OBAs
    GROUP BY AU_ID
)

SELECT
    a.BusinessID,
    a.AU_Name,
    a.Business_Segment,
    'EMP07' AS QuestionID,
    COALESCE(CAST(e.Included_Employee_Count AS STRING), '0') AS Response,
    COALESCE(e.Included_Employee_Count, 0) AS Included_Employee_Count,
    COALESCE(e.Flagged_Source_List, '') AS Flagged_Source_List,
    COALESCE(cc.Normalized_Cost_Centers, '') AS Normalized_Cost_Centers,
    CASE
        WHEN COALESCE(e.Included_Employee_Count, 0) > 0 THEN 'Mapped flagged OBA employees found for this AU.'
        ELSE 'No mapped flagged OBA employees found for this AU.'
    END AS Debug_Reason,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN AU_Employee_Counts e
    ON a.BusinessID = e.AU_ID
LEFT JOIN AU_Cost_Center_List cc
    ON a.BusinessID = cc.AU_ID
ORDER BY a.BusinessID;
